In [ ]:
# =====================================================
# PORT CONTAINER STACKING - Q LEARNING
# =====================================================

import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import imageio.v2 as imageio

from IPython.display import Image, display

# =====================================================
# GRID
# =====================================================

GRID_SIZE = 10
STACK_COLS = 4
MAX_STACK_HEIGHT = 3

# =====================================================
# BLOCKED STACK AREA
# üst 3 ve alt 3 satır bloklu
# =====================================================

BLOCKED_CELLS = set()

for r in range(3):

    for c in range(STACK_COLS):

        BLOCKED_CELLS.add((r,c))

for r in range(7,10):

    for c in range(STACK_COLS):

        BLOCKED_CELLS.add((r,c))

# =====================================================
# ACTIONS
# =====================================================

UP = 0
DOWN = 1
RIGHT = 2
LEFT = 3
PICKUP = 4
DROPOFF = 5

ACTIONS = [
    "UP",
    "DOWN",
    "RIGHT",
    "LEFT",
    "PICKUP",
    "DROPOFF"
]

ACTION_SIZE = 6

# =====================================================
# CONTAINERS
#
# priority:
#
# 0 -> YESIL
# 1 -> SARI
# 2 -> KIRMIZI
# =====================================================

CONTAINERS = {

    # =================================================
    # YESIL
    # =================================================

    0: {"pickup": (0,4), "shape": (1,2), "priority": 0},
    1: {"pickup": (2,4), "shape": (1,3), "priority": 0},
    2: {"pickup": (4,4), "shape": (3,1), "priority": 0},
    3: {"pickup": (7,4), "shape": (2,1), "priority": 0},
    4: {"pickup": (0,8), "shape": (1,2), "priority": 0},
    5: {"pickup": (3,8), "shape": (2,1), "priority": 0},

    # =================================================
    # SARI
    # =================================================

    6: {"pickup": (1,7), "shape": (1,2), "priority": 1},
    7: {"pickup": (5,7), "shape": (1,3), "priority": 1},
    8: {"pickup": (2,9), "shape": (2,1), "priority": 1},
    9: {"pickup": (6,7), "shape": (3,1), "priority": 1},
    10: {"pickup": (9,7), "shape": (1,2), "priority": 1},
    11: {"pickup": (4,6), "shape": (3,1), "priority": 1},

    # =================================================
    # KIRMIZI
    # =================================================

    12: {"pickup": (1,4), "shape": (1,2), "priority": 2},
    13: {"pickup": (3,5), "shape": (1,3), "priority": 2},
    14: {"pickup": (5,5), "shape": (2,1), "priority": 2},
    15: {"pickup": (8,5), "shape": (2,1), "priority": 2},
    16: {"pickup": (8,8), "shape": (1,2), "priority": 2},
    17: {"pickup": (0,6), "shape": (2,1), "priority": 2}

}

# =====================================================
# RANDOM 12 SIPARIS
# =====================================================

ORDER_CONTAINERS = random.sample(
    list(CONTAINERS.keys()),
    12
)

print("\nSIPARIS LISTESI:\n")

for cid in sorted(ORDER_CONTAINERS):

    print(f"C{cid+1}")

# =====================================================
# COLORS
# =====================================================

PRIORITY_COLORS = {

    0: "#00aa00",
    1: "#ffcc00",
    2: "#cc0000"

}

HEIGHT_COLORS = {

    1: None,
    2: "#8000ff",
    3: "#000000"

}

# =====================================================
# ENCODE
# =====================================================

def encode(row,col,carrying,delivered_mask):

    return (

        row * 10 * 32 * (2**18) +

        col * 32 * (2**18) +

        carrying * (2**18) +

        delivered_mask

    )

# =====================================================
# DECODE
# =====================================================

def decode(state):

    delivered_mask = state % (2**18)

    state //= (2**18)

    carrying = state % 32

    state //= 32

    col = state % 10

    row = state // 10

    return row,col,carrying,delivered_mask

# =====================================================
# ENVIRONMENT
# =====================================================

class PortEnv:

    def reset(self):

        self.row = 0
        self.col = 0

        self.carrying = 31

        self.delivered_mask = 0

        self.height_grid = np.zeros((10,4))

        self.priority_grid = np.full((10,4), -1)

        self.placed_containers = []

        self.current_target = None

        return encode(

            self.row,
            self.col,
            self.carrying,
            self.delivered_mask

        )

    # =================================================

    def is_delivered(self,cid):

        return (
            self.delivered_mask & (1 << cid)
        ) != 0

    # =================================================

    def mark_delivered(self,cid):

        self.delivered_mask |= (1 << cid)

    # =================================================

    def all_delivered(self):

        for cid in ORDER_CONTAINERS:

            if not self.is_delivered(cid):

                return False

        return True

    # =================================================

    def any_valid_moves_left(self):

        for cid in ORDER_CONTAINERS:

            if self.is_delivered(cid):
                continue

            pos = self.find_stack_position(cid)

            if pos is not None:

                return True

        return False

    # =================================================
    # STACK POSITION
    # =====================================================

    def find_stack_position(self,cid):

        h,w = CONTAINERS[cid]["shape"]

        new_priority = CONTAINERS[cid]["priority"]

        for target_height in range(MAX_STACK_HEIGHT):

            for r in range(10):

                for c in range(STACK_COLS):

                    if c + w > STACK_COLS:
                        continue

                    if r + h > 10:
                        continue

                    heights = []
                    priorities = []

                    valid = True

                    for i in range(h):

                        for j in range(w):

                            if (r+i,c+j) in BLOCKED_CELLS:

                                valid = False

                            heights.append(
                                self.height_grid[r+i][c+j]
                            )

                            priorities.append(
                                self.priority_grid[r+i][c+j]
                            )

                    if not valid:
                        continue

                    if len(set(heights)) != 1:
                        continue

                    current_height = heights[0]

                    if current_height != target_height:
                        continue

                    if current_height >= MAX_STACK_HEIGHT:
                        continue

                    for p in priorities:

                        if p == -1:
                            continue

                        if new_priority < p:

                            valid = False

                    if not valid:
                        continue

                    return (r,c)

        return None

    # =================================================
    # PLACE CONTAINER
    # =====================================================

    def place_container(
        self,
        cid,
        crane_row,
        crane_col
    ):

        target = self.current_target

        if target is None:

            return False,-100

        tr,tc = target

        if crane_row != tr or crane_col != tc:

            return False,-100

        h,w = CONTAINERS[cid]["shape"]

        priority = CONTAINERS[cid]["priority"]

        old_height = self.height_grid[tr][tc]

        for i in range(h):

            for j in range(w):

                self.height_grid[
                    tr+i
                ][
                    tc+j
                ] += 1

                self.priority_grid[
                    tr+i
                ][
                    tc+j
                ] = priority

        self.placed_containers.append({

            "cid": cid,
            "row": tr,
            "col": tc,
            "h": h,
            "w": w,
            "height": old_height + 1

        })

        reward_bonus = 0

        if old_height + 1 == 2:

            reward_bonus += 150

        elif old_height + 1 == 3:

            reward_bonus += 300

        return True,reward_bonus

    # =================================================
    # STEP
    # =====================================================

    def step(self,state,action):

        row,col,carrying,delivered = decode(state)

        self.row = row
        self.col = col
        self.carrying = carrying
        self.delivered_mask = delivered

        reward = -10

        # =================================================
        # MOVE
        # =====================================================

        if action == UP:

            old = self.row

            self.row = max(0,self.row-1)

            if old == self.row:

                reward -= 50

        elif action == DOWN:

            old = self.row

            self.row = min(9,self.row+1)

            if old == self.row:

                reward -= 50

        elif action == RIGHT:

            old = self.col

            self.col = min(9,self.col+1)

            if old == self.col:

                reward -= 50

        elif action == LEFT:

            old = self.col

            self.col = max(0,self.col-1)

            if old == self.col:

                reward -= 50

        # =================================================
        # PICKUP
        # =====================================================

        elif action == PICKUP:

            if self.carrying != 31:

                reward = -50

            else:

                success = False

                # =============================================
                # PRIORITY KONTROL
                # =============================================

                remaining_green = False
                remaining_yellow = False

                for cid_check in ORDER_CONTAINERS:

                    if self.is_delivered(cid_check):
                        continue

                    prio = CONTAINERS[
                        cid_check
                    ]["priority"]

                    if prio == 0:

                        remaining_green = True

                    elif prio == 1:

                        remaining_yellow = True

                # =============================================
                # HANGI PRIORITY ALINABILIR
                # =============================================

                allowed_priority = 2

                if remaining_green:

                    allowed_priority = 0

                elif remaining_yellow:

                    allowed_priority = 1

                # =============================================
                # SADECE O PRIORITY
                # =============================================

                sorted_containers = sorted(

                    [

                        (cid, CONTAINERS[cid])

                        for cid in ORDER_CONTAINERS

                        if (
                            CONTAINERS[cid]["priority"]
                            == allowed_priority
                        )

                    ],

                    key=lambda x: x[0]

                )

                # =============================================
                # PICKUP
                # =============================================

                for cid,data in sorted_containers:

                    if self.is_delivered(cid):
                        continue

                    pr,pc = data["pickup"]

                    if (self.row,self.col) == (pr,pc):

                        target = self.find_stack_position(cid)

                        if target is None:

                            if not self.any_valid_moves_left():

                                reward = -1000

                                done = True

                                next_state = encode(

                                    self.row,
                                    self.col,
                                    self.carrying,
                                    self.delivered_mask

                                )

                                return next_state,reward,done

                            else:

                                reward = -300

                                success = True

                                break

                        self.current_target = target

                        self.carrying = cid

                        priority = data["priority"]

                        if priority == 0:

                            reward = 1200

                        elif priority == 1:

                            reward = 700

                        else:

                            reward = 300

                        success = True

                        break

                if not success:

                    reward = -50

        # =================================================
        # DROPOFF
        # =====================================================

        elif action == DROPOFF:

            if self.carrying == 31:

                reward = -50

            else:

                success,bonus = self.place_container(

                    self.carrying,
                    self.row,
                    self.col

                )

                if success:

                    priority = CONTAINERS[
                        self.carrying
                    ]["priority"]

                    self.mark_delivered(
                        self.carrying
                    )

                    if priority == 0:

                        reward = 1000

                    elif priority == 1:

                        reward = 600

                    else:

                        reward = 250

                    reward += bonus

                    self.carrying = 31

                    self.current_target = None

                else:

                    reward = -150

        # =================================================
        # DONE
        # =====================================================

        done = self.all_delivered()

        if done:

            reward += 5000

        if not done:

            if not self.any_valid_moves_left():

                reward -= 1000

                done = True

        next_state = encode(

            self.row,
            self.col,
            self.carrying,
            self.delivered_mask

        )

        return next_state,reward,done

# =====================================================
# TRAIN
# =====================================================

env = PortEnv()

Q = {}

alpha = 0.1
gamma = 0.9
epsilon = 1.0

episodes = 20000

episode_rewards = []

# =====================================================

def get_q(state,action):

    key = (state,action)

    if key not in Q:

        Q[key] = 0

    return Q[key]

# =====================================================

for ep in range(episodes):

    state = env.reset()

    done = False

    step_count = 0

    total_reward = 0

    action_history = []

    position_history = []

    while not done and step_count < 300:

        if random.random() < epsilon:

            action = random.randint(0,5)

        else:

            q_values = [

                get_q(state,a)

                for a in range(ACTION_SIZE)

            ]

            action = np.argmax(q_values)

        next_state,reward,done = env.step(

            state,
            action

        )

        total_reward += reward

        # =================================================
        # LOOP CONTROL
        # =====================================================

        nr,nc,_,_ = decode(next_state)

        position_history.append((nr,nc))

        if len(position_history) > 15:

            position_history.pop(0)

        same_count = position_history.count((nr,nc))

        if same_count >= 6:

            reward -= 2000

            done = True

        if len(position_history) >= 10:

            unique_positions = len(
                set(position_history[-10:])
            )

            if unique_positions <= 3:

                reward -= 150

        # =================================================
        # ACTION SPAM CEZASI
        # =====================================================

        action_history.append(action)

        if len(action_history) > 8:

            action_history.pop(0)

        if len(action_history) >= 4:

            last4 = action_history[-4:]

            if last4 == [UP,DOWN,UP,DOWN]:

                reward -= 120

            elif last4 == [DOWN,UP,DOWN,UP]:

                reward -= 120

            elif last4 == [LEFT,RIGHT,LEFT,RIGHT]:

                reward -= 120

            elif last4 == [RIGHT,LEFT,RIGHT,LEFT]:

                reward -= 120

        # =================================================
        # Q UPDATE
        # =====================================================

        old_q = get_q(state,action)

        next_max = max([

            get_q(next_state,a)

            for a in range(ACTION_SIZE)

        ])

        new_q = old_q + alpha * (

            reward +
            gamma * next_max -
            old_q

        )

        Q[(state,action)] = new_q

        state = next_state

        step_count += 1

    episode_rewards.append(total_reward)

    epsilon = max(0.01,epsilon*0.995)

    if ep % 500 == 0:

        print(
            f"Episode: {ep} | "
            f"Epsilon: {round(epsilon,3)}"
        )

print("\nEGITIM TAMAMLANDI")

# =====================================================
# TRAINING GRAPH
# =====================================================

plt.figure(figsize=(12,5))

plt.plot(episode_rewards)

plt.title("Training Reward Graph")

plt.xlabel("Episode")

plt.ylabel("Total Reward")

plt.grid()

plt.show()

# =====================================================
# RENDER
# =====================================================

def render_image(state,step):

    row,col,carrying,delivered = decode(state)

    fig,ax = plt.subplots(figsize=(10,10))

    ax.set_xlim(0,10)
    ax.set_ylim(0,10)

    ax.set_xticks(np.arange(0,11))
    ax.set_yticks(np.arange(0,11))

    ax.grid()

    # GRID

    for r in range(10):

        for c in range(10):

            if (r,c) in BLOCKED_CELLS:

                color = "#555555"

            elif c < STACK_COLS:

                color = "#ccffcc"

            else:

                color = "#ffcccc"

            rect = patches.Rectangle(

                (c,9-r),
                1,
                1,

                facecolor=color

            )

            ax.add_patch(rect)

    # BASLANGIC KONTEYNIRLARI

    for cid,data in CONTAINERS.items():

        if (delivered & (1 << cid)) != 0:
            continue

        if carrying == cid:
            continue

        pr,pc = data["pickup"]

        h,w = data["shape"]

        priority = data["priority"]

        color = PRIORITY_COLORS[
            priority
        ]

        rect = patches.Rectangle(

            (pc,9-pr-(h-1)),
            w,
            h,

            facecolor=color,
            edgecolor='black',
            linewidth=2

        )

        ax.add_patch(rect)

        txt_color = "white"

        if cid not in ORDER_CONTAINERS:

            txt_color = "black"

        ax.text(

            pc + 0.5,
            9-pr + 0.5,

            f"C{cid+1}",

            fontsize=8,
            ha='center',
            va='center',
            fontweight='bold',
            color=txt_color

        )

    # ISTIFLENENLER

    for item in env.placed_containers:

        cid = item["cid"]

        r = item["row"]
        c = item["col"]

        h = item["h"]
        w = item["w"]

        height = item["height"]

        priority = CONTAINERS[cid]["priority"]

        if height == 1:

            color = PRIORITY_COLORS[
                priority
            ]

        else:

            color = HEIGHT_COLORS[
                height
            ]

        rect = patches.Rectangle(

            (c,9-r-h+1),
            w,
            h,

            facecolor=color,
            edgecolor='black',
            linewidth=2,
            alpha=0.9

        )

        ax.add_patch(rect)

        ax.text(

            c + 0.5,
            9-r + 0.5,

            f"C{cid+1}",

            fontsize=8,
            ha='center',
            va='center',
            fontweight='bold',
            color='white'

        )

    # VINC

    if carrying != 31:

        crane_text = f"V\nC{carrying+1}"

    else:

        crane_text = "V"

    ax.text(

        col+0.5,
        9-row+0.5,

        crane_text,

        fontsize=10,
        ha='center',
        va='center',
        color='black',
        fontweight='bold'

    )

    ax.set_title(
        f"STEP: {step}"
    )

    # SIPARIS YAZISI

    order_text = "SIPARIS: "

    for cid in sorted(ORDER_CONTAINERS):

        order_text += f"C{cid+1} "

    fig.text(

        0.5,
        0.02,

        order_text,

        ha='center',
        fontsize=10,
        fontweight='bold'

    )

    fig.canvas.draw()

    image = np.asarray(
        fig.canvas.buffer_rgba()
    )

    plt.close(fig)

    return image

# =====================================================
# TEST
# =====================================================

epsilon = 0

frames = []

state = env.reset()

done = False

step = 0

frames.append(
    render_image(state,step)
)

print("\nTEST BASLIYOR\n")

while not done and step < 300:

    q_values = [

        get_q(state,a)

        for a in range(ACTION_SIZE)

    ]

    action = np.argmax(q_values)

    next_state,reward,done = env.step(

        state,
        action

    )

    step += 1

    print(

        f"STEP: {step} | "
        f"ACTION: {ACTIONS[action]} | "
        f"REWARD: {reward}"

    )

    frames.append(
        render_image(next_state,step)
    )

    state = next_state

print("\nTEST BITTI")

# =====================================================
# GIF
# =====================================================

imageio.mimsave(

    "vinc_final.gif",
    frames,
    fps=3

)

print("\nGIF olusturuldu")

display(
    Image(filename="vinc_final.gif")
)